In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
pd.set_option('display.max_columns', None)  # so wide tables don't get truncated when we look at them

In [ ]:
df = pd.read_csv('/content/Customer-Churn.csv')

print("Shape:", df.shape)
df.info()

Shape: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 n

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print(churn_counts)
print(churn_pct)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [ ]:
telco_data = df.copy()  # work on a copy; keep the original df untouched in case we need to compare back to it

telco_data['TotalCharges'] = pd.to_numeric(telco_data['TotalCharges'], errors='coerce')

print("Missing values created:", telco_data['TotalCharges'].isnull().sum())

Missing values created: 11


In [ ]:
telco_data[telco_data['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']]
#From this, observing tenure = 0 -> it's evident that these are new customers who've just started, so haven't been charged yet. Ofc
#the amount/ Total charges = 0.

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


In [ ]:
telco_data = telco_data.dropna(subset=['TotalCharges'])
print(telco_data.shape)

(7032, 21)


In [ ]:
# No transformation needed for modeling — tenure stays as a continuous column.
# (tenure_bin above was only for the churn-rate-by-cohort analysis, not used as a feature.)
print(telco_data['tenure'].describe())

count    7032.000000
mean       32.421786
std        24.545260
min         1.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64


In [ ]:
X = telco_data.drop(columns=['customerID', 'Churn'])
y = telco_data['Churn'].map({'No': 0, 'Yes': 1})

X = pd.get_dummies(X, drop_first=True)

print("Shape of X after encoding:", X.shape)
X.head()

Shape of X after encoding: (7032, 30)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

Train shape: (5625, 30)  Test shape: (1407, 30)
Train churn rate: 0.2657777777777778
Test churn rate: 0.2658137882018479


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

print("Tree depth:", model_dt.get_depth())
print("Number of leaves:", model_dt.get_n_leaves())

Tree depth: 23
Number of leaves: 1112


In [ ]:
train_acc = accuracy_score(y_train, model_dt.predict(X_train))
test_acc = accuracy_score(y_test, model_dt.predict(X_test))

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")

Train accuracy: 0.9988
Test accuracy:  0.7186


In [ ]:
model_dt_weighted = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
model_dt_weighted.fit(X_train, y_train)

print(classification_report(y_test, model_dt_weighted.predict(X_test)))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [ ]:
# Cell 14 — SMOTE, and a live check for the bool-casting corruption
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print("Original train shape:", X_train.shape, dict(y_train.value_counts()))
print("After SMOTE:         ", X_train_sm.shape, dict(y_train_sm.value_counts()))

print("\ngender_Male true rate (original train):", X_train['gender_Male'].mean().round(4))
print("gender_Male rate among SMOTE-synthetic rows:",
      X_train_sm['gender_Male'].iloc[X_train.shape[0]:].mean().round(4))

Original train shape: (5625, 30) {0: np.int64(4130), 1: np.int64(1495)}
After SMOTE:          (8260, 30) {0: np.int64(4130), 1: np.int64(4130)}

gender_Male true rate (original train): 0.5019
gender_Male rate among SMOTE-synthetic rows: 0.7218


In [ ]:
# Cell 15 — SMOTENC: the categorical-aware fix
from imblearn.over_sampling import SMOTENC

continuous_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_mask = [col not in continuous_cols for col in X_train.columns]

smnc = SMOTENC(categorical_features=categorical_mask, random_state=42)
X_train_smnc, y_train_smnc = smnc.fit_resample(X_train, y_train)

print("After SMOTENC:", X_train_smnc.shape, dict(y_train_smnc.value_counts()))
print("gender_Male rate among SMOTENC-synthetic rows:",
      X_train_smnc['gender_Male'].iloc[X_train.shape[0]:].mean().round(4))

After SMOTENC: (8260, 30) {0: np.int64(4130), 1: np.int64(4130)}
gender_Male rate among SMOTENC-synthetic rows: 0.5051


In [ ]:
# Cell 16 — train the same (unpruned, default) tree config as Cell 10 on the SMOTENC data,
# evaluate on the untouched, never-resampled test set
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

clf_smnc = DecisionTreeClassifier(random_state=42)
clf_smnc.fit(X_train_smnc, y_train_smnc)
print(classification_report(y_test, clf_smnc.predict(X_test)))

              precision    recall  f1-score   support

           0       0.84      0.79      0.82      1033
           1       0.50      0.58      0.54       374

    accuracy                           0.74      1407
   macro avg       0.67      0.69      0.68      1407
weighted avg       0.75      0.74      0.74      1407



In [ ]:
# Cell 17 — same pruning as Cell 13 (depth=6), but on SMOTENC-resampled data instead of
# class_weight='balanced' -- isolates the resampling effect from the pruning effect
clf_smnc_pruned = DecisionTreeClassifier(max_depth=6, random_state=42)
clf_smnc_pruned.fit(X_train_smnc, y_train_smnc)
print(classification_report(y_test, clf_smnc_pruned.predict(X_test)))

              precision    recall  f1-score   support

           0       0.87      0.80      0.83      1033
           1       0.54      0.67      0.60       374

    accuracy                           0.76      1407
   macro avg       0.71      0.73      0.72      1407
weighted avg       0.78      0.76      0.77      1407



In [ ]:
# Cell 18 — RandomForest: the concrete bagging example, on the SMOTENC-resampled data
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_smnc, y_train_smnc)

print("RF train accuracy:", round(rf.score(X_train_smnc, y_train_smnc), 4))
print(classification_report(y_test, rf.predict(X_test)))

RF train accuracy: 0.9992
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      1033
           1       0.57      0.61      0.59       374

    accuracy                           0.77      1407
   macro avg       0.71      0.72      0.71      1407
weighted avg       0.78      0.77      0.77      1407



In [ ]:
# Cell 19 — faithful reproduction of the deployed model's actual training setup:
# AdaBoost + manual sample_weight (equivalent to class_weight='balanced'), on ORIGINAL train data
from sklearn.ensemble import AdaBoostClassifier
import numpy as np

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
weight_positive = negative_count / positive_count
print("weight_positive:", round(weight_positive, 3))  # should land at 2.763, matching the balanced-weight ratio

sample_weight = np.where(y_train == 1, weight_positive, 1.0)

ada = AdaBoostClassifier(n_estimators=50, random_state=42)
ada.fit(X_train, y_train, sample_weight=sample_weight)

print(classification_report(y_test, ada.predict(X_test)))

weight_positive: 2.763
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.73      0.74      1407



In [ ]:
# Cell 20 — same AdaBoost architecture, swap weighting for resampling
ada_smnc = AdaBoostClassifier(n_estimators=50, random_state=42)
ada_smnc.fit(X_train_smnc, y_train_smnc)
print(classification_report(y_test, ada_smnc.predict(X_test)))

              precision    recall  f1-score   support

           0       0.89      0.75      0.81      1033
           1       0.52      0.74      0.61       374

    accuracy                           0.75      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.79      0.75      0.76      1407



In [ ]:
# Cell 21 — XGBoost, native imbalance handling via scale_pos_weight
from xgboost import XGBClassifier

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight:", round(scale_pos_weight, 3))  # same 2.763 ratio as before

xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

print("XGBoost train accuracy:", round(xgb.score(X_train, y_train), 4))
print(classification_report(y_test, xgb.predict(X_test)))

scale_pos_weight: 2.763
XGBoost train accuracy: 0.9307
              precision    recall  f1-score   support

           0       0.86      0.77      0.81      1033
           1       0.51      0.66      0.58       374

    accuracy                           0.74      1407
   macro avg       0.69      0.72      0.70      1407
weighted avg       0.77      0.74      0.75      1407



In [ ]:
# Cell 22 — RandomizedSearchCV: proper tuning via cross-validation, test set untouched
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth': [3, 4, 5, 6, 8],
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0]
}

xgb_base = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')

search = RandomizedSearchCV(
    xgb_base, param_distributions=param_dist,
    n_iter=20, scoring='f1', cv=5, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))

best_xgb = search.best_estimator_
print(classification_report(y_test, best_xgb.predict(X_test)))

Best params: {'subsample': 0.8, 'n_estimators': 50, 'max_depth': 8, 'learning_rate': 0.05}
Best CV F1: 0.6325
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1033
           1       0.53      0.78      0.63       374

    accuracy                           0.76      1407
   macro avg       0.72      0.76      0.72      1407
weighted avg       0.80      0.76      0.77      1407



In [ ]:
# Cell 23 — raw (pre-encoding) feature set, split identically to X_train/X_test
X_raw = telco_data.drop(columns=['customerID', 'Churn'])

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train_raw.shape, X_test_raw.shape)
print("Same customers as before:", (X_train_raw.index == X_train.index).all())

(5625, 19) (1407, 19)
Same customers as before: True


In [ ]:
# Cell 24 — preprocessing + tuned model as one saved, deployable object
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib

categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                     'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                     'PaperlessBilling', 'PaymentMethod']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
], remainder='passthrough')

final_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', XGBClassifier(**search.best_params_, scale_pos_weight=scale_pos_weight,
                             random_state=42, eval_metric='logloss'))
])

final_pipeline.fit(X_train_raw, y_train)
print(classification_report(y_test, final_pipeline.predict(X_test_raw)))

joblib.dump(final_pipeline, 'churn_pipeline.pkl')
print("Saved.")

              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1033
           1       0.53      0.78      0.63       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.77      1407

Saved.


In [ ]:
# Cell 25 — optional proof: reload the saved pipeline and score a raw, unencoded customer
loaded_model = joblib.load('churn_pipeline.pkl')
print(loaded_model.predict(X_test_raw.iloc[[0]]))
print(X_test_raw.iloc[[0]])
print("Predicted churn:", loaded_model.predict(X_test_raw.iloc[[0]])[0])

[0]
     gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
974  Female              0     Yes        Yes      59          Yes   

    MultipleLines InternetService OnlineSecurity OnlineBackup  \
974            No             DSL             No          Yes   

    DeviceProtection TechSupport StreamingTV StreamingMovies  Contract  \
974               No         Yes         Yes             Yes  Two year   

    PaperlessBilling            PaymentMethod  MonthlyCharges  TotalCharges  
974              Yes  Credit card (automatic)           75.95       4542.35  
Predicted churn: 0
